# Fashion Recommendation with Content Features
## Hybrid Model: Sequence + Caption/Hashtag/Mention Features

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import matplotlib.pyplot as plt
import ast
import re
import os
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [2]:
df_seq = pd.read_csv('input/user_behavior.csv')

if isinstance(df_seq['posts_sequence'].iloc[0], str):
    df_seq['posts_sequence_list'] = df_seq['posts_sequence'].apply(ast.literal_eval)
else:
    df_seq['posts_sequence_list'] = df_seq['posts_sequence']

# Parse interaction_sequence to extract polarity scores
def parse_interaction_sequence(seq_str):
    """Parse interaction_sequence string to extract post_id -> polarity mapping"""
    try:
        seq = eval(seq_str, {"Timestamp": pd.Timestamp})
        return {item['post_id']: item.get('polarity', 0.5) for item in seq}
    except:
        return {}

df_seq['polarity_map'] = df_seq['interaction_sequence'].apply(parse_interaction_sequence)
print(f'Parsed polarity for {len(df_seq)} users')

# Sentiment-based weight function
def polarity_to_weight(polarity):
    """Convert polarity score to training weight.
    Positive sentiment (polarity >= 0.5): weight = 1.0 (emphasize learning)
    Neutral sentiment (0.0 <= polarity < 0.5): weight = 0.7
    Negative sentiment (polarity < 0.0): weight = 0.3 (down-weight)
    """
    if polarity >= 0.5:
        return 1.0
    elif polarity >= 0.0:
        return 0.7
    else:
        return 0.3

all_post_ids = set()
for seq in df_seq['posts_sequence_list']:
    all_post_ids.update(seq)

sorted_ids = sorted(list(all_post_ids))

post2idx = {pid: i+1 for i, pid in enumerate(sorted_ids)}
idx2post = {i+1: pid for i, pid in enumerate(sorted_ids)}
VOCAB_SIZE = len(post2idx) + 1

print(f"Vocab Size: {VOCAB_SIZE}")

Parsed polarity for 231 users
Vocab Size: 530


In [3]:
df_posts = pd.read_csv('input/content.csv')
def prepare_text(row):
    cap = str(row.get('caption', '')) if not pd.isna(row.get('caption')) else ""
    hash_ = str(row.get('hashtags', '')) if not pd.isna(row.get('hashtags')) else ""
    ment_ = str(row.get('mentions', '')) if not pd.isna(row.get('mentions')) else ""
    
    # Ghép chuỗi với token đặc biệt [SEP]
    text = f"[CLS] {cap} [SEP] {hash_} [SEP] {ment_} [SEP]"
    text = re.sub(r"[\[\]']", "", text) 
    return text

df_posts['text_content'] = df_posts.apply(prepare_text, axis=1)
id_to_text = pd.Series(df_posts.text_content.values, index=df_posts.post_id).to_dict()

In [4]:
EMBEDDING_FILE = 'post_embeddings_text_only.npy'

if os.path.exists(EMBEDDING_FILE):
    print(f"🔹 Đang tải embedding đã có từ file {EMBEDDING_FILE}...")
    pretrained_weights = np.load(EMBEDDING_FILE)
else:
    print("⏳ Đang tạo BERT embedding mới (Chỉ văn bản)...")
    
    # Khởi tạo BERT
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
    bert_model.eval()
    
    # Tạo ma trận rỗng [Vocab_Size, 768]
    pretrained_weights = np.zeros((VOCAB_SIZE, 768))
    
    batch_size = 32
    post_indices = list(range(1, VOCAB_SIZE)) # Index từ 1 đến N
    
    with torch.no_grad():
        for i in tqdm(range(0, len(post_indices), batch_size)):
            batch_idx = post_indices[i : i + batch_size]
            
            # Lấy text cho batch này
            batch_text = []
            for idx in batch_idx:
                pid = idx2post[idx] # Lấy ID thực
                text = id_to_text.get(pid, "[CLS] [SEP]") # Lấy text, mặc định nếu thiếu
                batch_text.append(text)
            
            # Tokenize & Đưa qua model
            inputs = tokenizer(batch_text, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
            outputs = bert_model(**inputs)
            cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy() # Lấy vector [CLS]
            
            # Điền vào ma trận
            for j, idx in enumerate(batch_idx):
                pretrained_weights[idx] = cls_emb[j]
                
    # Lưu file để dùng lại lần sau
    np.save(EMBEDDING_FILE, pretrained_weights)
    print("✅ Đã tạo và lưu file embedding.")

print(f"Kích thước ma trận Embedding: {pretrained_weights.shape}")

⏳ Đang tạo BERT embedding mới (Chỉ văn bản)...



  0%|          | 0/17 [00:00<?, ?it/s]

C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\venv\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(

  6%|▌         | 1/17 [00:00<00:07,  2.18it/s]


 12%|█▏        | 2/17 [00:00<00:04,  3.27it/s]


 18%|█▊        | 3/17 [00:00<00:03,  3.87it/s]

...


 88%|████████▊ | 15/17 [00:03<00:00,  4.68it/s]


 94%|█████████▍| 16/17 [00:03<00:00,  4.61it/s]


100%|██████████| 17/17 [00:03<00:00,  5.30it/s]


100%|██████████| 17/17 [00:03<00:00,  4.70it/s]

✅ Đã tạo và lưu file embedding.
Kích thước ma trận Embedding: (530, 768)


In [5]:
train_samples = []
test_samples = []

for idx, row in df_seq.iterrows():
    raw_seq = row['posts_sequence_list']
    polarity_map = row['polarity_map']
    seq = [post2idx.get(pid, 0) for pid in raw_seq]
    
    if len(seq) < 2: continue 
    
    # Test sample: use average polarity of sequence as weight
    avg_polarity = np.mean([polarity_map.get(pid, 0.5) for pid in raw_seq])
    test_weight = polarity_to_weight(avg_polarity)
    test_samples.append((seq[:-1], seq[-1], test_weight))
    
    # Train samples: use polarity of target item
    train_seq_full = seq[:-1]
    for i in range(1, len(train_seq_full)):
        start = max(0, i - 20)
        target_pid = raw_seq[i]
        polarity = polarity_map.get(target_pid, 0.5)
        weight = polarity_to_weight(polarity)
        train_samples.append((train_seq_full[start:i], train_seq_full[i], weight))

print(f"Train Samples: {len(train_samples)}")
print(f"Test Samples: {len(test_samples)}")

# Show weight distribution
train_weights = [s[2] for s in train_samples]
print(f'Weight distribution: 1.0={train_weights.count(1.0)}, 0.7={train_weights.count(0.7)}, 0.3={train_weights.count(0.3)}')

Train Samples: 658
Test Samples: 231
Weight distribution: 1.0=249, 0.7=309, 0.3=100


In [6]:
class InteractionDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        seq, target, weight = self.samples[idx]
        return (torch.tensor(seq, dtype=torch.long), 
                torch.tensor(target, dtype=torch.long),
                torch.tensor(weight, dtype=torch.float))

def collate_fn(batch):
    inputs, targets, weights = zip(*batch)
    inputs_padded = torch.nn.utils.rnn.pad_sequence(inputs, batch_first=True, padding_value=0)
    return inputs_padded, torch.stack(targets), torch.stack(weights)

BATCH_SIZE = 32
train_loader = DataLoader(InteractionDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(InteractionDataset(test_samples), batch_size=BATCH_SIZE, collate_fn=collate_fn)

print(f"Số mẫu Train: {len(train_samples)} | Số mẫu Test: {len(test_samples)}")

Số mẫu Train: 658 | Số mẫu Test: 231


In [7]:
class MHA_Content_Model(nn.Module):
    def __init__(self, pretrained_weights, hidden_dim=64, num_layers=1, num_heads=4, dropout=0.5):
        super(MHA_Content_Model, self).__init__()
        
        # 1. Tải trọng số Pre-trained (Đóng băng - Frozen)
        weights_tensor = torch.FloatTensor(pretrained_weights)
        # padding_idx=0 để vector tại index 0 luôn là vector 0
        self.embedding = nn.Embedding.from_pretrained(weights_tensor, freeze=True, padding_idx=0)
        
        bert_dim = weights_tensor.shape[1] # 768 chiều từ BERT
        vocab_size = weights_tensor.shape[0]
        
        # 2. Lớp Projection (Quan trọng: Nén 768 -> 64)
        self.projection = nn.Sequential(
            nn.Linear(bert_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 3. BiLSTM
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        
        # 4. Multi-Head Attention (Đầu vào là Hidden * 2 do BiLSTM)
        self.mha = nn.MultiheadAttention(embed_dim=hidden_dim*2, num_heads=num_heads, 
                                         dropout=dropout, batch_first=True)
        
        self.norm = nn.LayerNorm(hidden_dim*2)
        self.dropout = nn.Dropout(dropout)
        
        # 5. Lớp Output
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)
        
    def forward(self, x):
        # x: [Batch, Seq]
        
        # A. Embedding & Projection
        embedded = self.embedding(x) # [B, L, 768]
        proj = self.projection(embedded) # [B, L, 64]
        
        # B. BiLSTM
        lstm_out, _ = self.lstm(proj) # [B, L, 128]
        
        # C. MHA & Residual Connection
        attn_out, _ = self.mha(lstm_out, lstm_out, lstm_out)
        out = self.norm(attn_out + lstm_out)
        
        # D. Mean Pooling (Lấy đặc trưng trung bình)
        # Tạo mask để không tính padding (số 0) vào trung bình
        mask = (x != 0).unsqueeze(-1).float()
        out = out * mask
        sum_out = torch.sum(out, dim=1)
        count = torch.sum(mask, dim=1)
        context = sum_out / (count + 1e-9) # Tránh chia cho 0
        
        # E. Predict
        logits = self.fc(self.dropout(context))
        return logits

In [8]:
def train_model(model, train_loader, epochs=50, lr=0.001):
    # Sử dụng weight_decay để giảm overfitting
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # Use reduction='none' to apply per-sample sentiment weights
    criterion = nn.CrossEntropyLoss(reduction='none')
    model.train()
    
    print(f"🚀 Bắt đầu huấn luyện với Sentiment Weighting...")
    for epoch in range(epochs):
        total_loss = 0
        for inputs, targets, weights in train_loader:
            inputs, targets, weights = inputs.to(device), targets.to(device), weights.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # Compute per-sample loss and apply sentiment weights
            per_sample_loss = criterion(outputs, targets)
            weighted_loss = (per_sample_loss * weights).mean()
            
            weighted_loss.backward()
            optimizer.step()
            total_loss += weighted_loss.item()
            
        if (epoch+1) % 10 == 0:
            print(f"   Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")
    return model

def evaluate(model, test_loader, k_list=[5, 10]):
    model.eval()
    hits = {k: 0 for k in k_list}
    mrr_sum = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets, _ in test_loader:  # Ignore weights during evaluation
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            
            # Lấy Top-20 dự đoán cao nhất
            _, topk = torch.topk(outputs, 20, dim=1)
            topk = topk.cpu().numpy()
            
            for i, target in enumerate(targets):
                true_id = target.item()
                pred_ids = topk[i]
                
                # Tính Hit Rate
                for k in k_list:
                    if true_id in pred_ids[:k]:
                        hits[k] += 1
                
                # Tính MRR
                if true_id in pred_ids:
                    rank = np.where(pred_ids == true_id)[0][0] + 1
                    mrr_sum += 1.0 / rank
                
                total += 1
                
    return {f'HR@{k}': v/total for k, v in hits.items()}, mrr_sum/total

In [9]:
HIDDEN_DIM = 256
LAYERS = 1
HEADS = 4
DROPOUT = 0.5
EPOCHS = 80
LR = 0.00015

model_text = MHA_Content_Model(pretrained_weights, 
                               hidden_dim=HIDDEN_DIM, 
                               num_layers=LAYERS, 
                               num_heads=HEADS, 
                               dropout=DROPOUT).to(device)

# 2. Huấn luyện
train_model(model_text, train_loader, epochs=EPOCHS, lr=LR)

# 3. Đánh giá
metrics_text, mrr_text = evaluate(model_text, test_loader)

print("\n" + "="*50)
print("📊 KẾT QUẢ THỰC NGHIỆM: MHA + NỘI DUNG VĂN BẢN")
print("="*50)
print(f"HR@5 : {metrics_text['HR@5']:.4f}")
print(f"HR@10: {metrics_text['HR@10']:.4f}")
print(f"MRR  : {mrr_text:.4f}")
print("="*50)

baseline_mrr = 0.1870 
improvement = ((mrr_text - baseline_mrr) / baseline_mrr) * 100

print(f"MRR Baseline (Chỉ dùng ID): {baseline_mrr:.4f}")
print(f"Độ cải thiện (Improvement): {improvement:+.2f}%")

🚀 Bắt đầu huấn luyện với Sentiment Weighting...


   Epoch 10/80 | Loss: 3.5353


   Epoch 20/80 | Loss: 2.5217


   Epoch 30/80 | Loss: 1.8775


   Epoch 40/80 | Loss: 1.4850


   Epoch 50/80 | Loss: 1.2074


   Epoch 60/80 | Loss: 1.0318


   Epoch 70/80 | Loss: 0.9307


   Epoch 80/80 | Loss: 0.7871

📊 KẾT QUẢ THỰC NGHIỆM: MHA + NỘI DUNG VĂN BẢN
HR@5 : 0.2597
HR@10: 0.3420
MRR  : 0.1941
MRR Baseline (Chỉ dùng ID): 0.1870
Độ cải thiện (Improvement): +3.79%
